# 03 — Feature Engineering
**E-Waste Toxic Gas Detection System — ML Pipeline**

**Purpose:** Analyse feature importance, assess dimensionality reduction (PCA), and confirm the final feature set.

**Author:** Sanjula Madushanka | Final Year Research Y4S2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR  = Path('../datasets')
SAVE_DIR  = Path('../results')
MODEL_DIR = Path('../models')

FEATURE_NAMES = ['mq2_ppm', 'mq7_ppm', 'mq135_ppm', 'mq303_ppm', 'mq136_ppm',
                 'temperature_c', 'humidity_pct']
SENSOR_LABELS = ['MQ-2\n(LPG)', 'MQ-7\n(CO)', 'MQ-135\n(VOC)', 'MQ-303\n(Hg)',
                 'MQ-136\n(H₂S)', 'Temp\n(°C)', 'Humidity\n(%)']

# Load preprocessed data
SPLIT = DATA_DIR / 'processed' / 'train_test_split'
X_train = pd.read_csv(SPLIT / 'X_train.csv').values
y_train = pd.read_csv(SPLIT / 'y_train.csv').values.ravel()
X_test  = pd.read_csv(SPLIT / 'X_test.csv').values
y_test  = pd.read_csv(SPLIT / 'y_test.csv').values.ravel()
le      = joblib.load(MODEL_DIR / 'label_encoder.pkl')

print(f'Data loaded: X_train={X_train.shape}, X_test={X_test.shape}')
print(f'Classes: {list(le.classes_)}')

## 3.1 Random Forest Feature Importance (Preliminary)

In [ ]:
# Quick RF to get feature importance (not the final tuned model)
rf_prelim = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_prelim.fit(X_train, y_train)

importances = rf_prelim.feature_importances_
indices     = np.argsort(importances)[::-1]  # Sorted high to low

print('=== FEATURE IMPORTANCE RANKING ===')
for rank, idx in enumerate(indices):
    bar = '█' * int(importances[idx] * 50)
    print(f'  {rank+1}. {FEATURE_NAMES[idx]:15s}: {importances[idx]:.4f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#ef4444' if i == indices[0] else '#3b82f6' if importances[i] > 0.10 else '#94a3b8'
          for i in range(len(FEATURE_NAMES))]

bars = ax.barh([SENSOR_LABELS[i] for i in indices],
               [importances[i] for i in indices],
               color=[colors[i] for i in indices],
               edgecolor='white', height=0.6)

# Add value labels
for bar, val in zip(bars, [importances[i] for i in indices]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('Feature Importance Score (Gini)', fontsize=11)
ax.set_title('Figure 6: Random Forest Feature Importance\n(Preliminary — 100 trees)',
             fontsize=13, fontweight='bold')
ax.axvline(1/len(FEATURE_NAMES), color='orange', linestyle='--', linewidth=1.5,
           label=f'Equal importance baseline (1/7 = {1/7:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

## 3.2 Feature Selection Decision

In [ ]:
# Threshold: keep features with importance > threshold
IMPORTANCE_THRESHOLD = 0.05  # Adjust if needed

selected_features = [FEATURE_NAMES[i] for i in range(len(FEATURE_NAMES))
                     if importances[i] >= IMPORTANCE_THRESHOLD]
dropped_features  = [FEATURE_NAMES[i] for i in range(len(FEATURE_NAMES))
                     if importances[i] < IMPORTANCE_THRESHOLD]

print(f'=== FEATURE SELECTION (threshold = {IMPORTANCE_THRESHOLD}) ===')
print(f'✅ Selected ({len(selected_features)}): {selected_features}')
print(f'❌ Dropped  ({len(dropped_features)}): {dropped_features}')

# Research decision: keep all 7 features for interpretability
# (removing temperature/humidity reduces physical meaning of the model)
print()
print('▶ Research Decision: KEEP ALL 7 FEATURES')
print('  Justification: Temperature and humidity are physical calibration variables.')
print('  Even with low importance, they prevent measurement errors at extreme conditions.')
print('  This is consistent with best practice in environmental sensor research.')

FINAL_FEATURES = FEATURE_NAMES  # All 7 kept
print(f'\nFinal feature set: {FINAL_FEATURES}')

## 3.3 PCA — Visualise Class Separability

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train)

print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.3f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.3f}')
print(f'Total variance explained: {sum(pca.explained_variance_ratio_):.3f}')

fig, ax = plt.subplots(figsize=(10, 7))
colors_map = {
    'CO': '#ef4444', 'LPG': '#f59e0b', 'BENZENE': '#8b5cf6',
    'AMMONIA': '#3b82f6', 'MERCURY': '#c0c0c0', 'H2S': '#22c55e', 'CLEAN': '#64748b'
}

for cls_idx, cls_name in enumerate(le.classes_):
    mask = y_train == cls_idx
    color = colors_map.get(cls_name, '#888888')
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=cls_name, alpha=0.6, s=40, color=color, edgecolors='white', linewidth=0.3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('Figure 7: PCA Projection — Gas Class Separability (2D)',
             fontsize=13, fontweight='bold')
ax.legend(title='Gas Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig7_pca_projection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')
print('\nNote: Well-separated clusters → good classification boundary → high expected accuracy')

## 3.4 Explained Variance Curve

In [ ]:
pca_full = PCA(n_components=X_train.shape[1], random_state=42)
pca_full.fit(X_train)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumulative_variance)+1), cumulative_variance * 100,
        'o-', color='#3b82f6', linewidth=2, markersize=8, markerfacecolor='white',
        markeredgewidth=2)
ax.axhline(95, color='red', linestyle='--', linewidth=1.5, label='95% threshold')
ax.axhline(99, color='orange', linestyle='--', linewidth=1.5, label='99% threshold')
ax.fill_between(range(1, len(cumulative_variance)+1), cumulative_variance * 100,
                alpha=0.1, color='#3b82f6')
for i, v in enumerate(cumulative_variance * 100):
    ax.annotate(f'{v:.1f}%', (i+1, v), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)
ax.set_xlabel('Number of Principal Components', fontsize=11)
ax.set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
ax.set_title('Figure 8: PCA Cumulative Explained Variance', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks(range(1, len(cumulative_variance)+1))
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig8_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')
print('\nResearch conclusion: All 7 features retained — PCA would reduce interpretability')

In [ ]:
print('=' * 60)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'Final feature count: {len(FINAL_FEATURES)}')
print(f'Features: {FINAL_FEATURES}')
print(f'PCA: NOT applied (all 7 features retained for interpretability)')
print(f'\nTop 3 most important features:')
for i, idx in enumerate(indices[:3]):
    print(f'  {i+1}. {FEATURE_NAMES[idx]} (importance = {importances[idx]:.4f})')
print()
print('✅ Notebook 03 complete — proceed to 04_model_training.ipynb')